In [2]:
#### READ FILES FROM METEOSWISS FOLDERS AND WRITE TXT FILES ####

# AE31 A11
# AE33
# AE31 S11
# Ozone tei49c

# used python version 3.8.10

In [3]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import doctest
import tarfile
from os.path import join, getsize
import matplotlib.ticker as ticker
from matplotlib import pyplot

In [4]:
#### AE31 FILES ####

def extract_file(file: str, sep=",", type=".tar.gz", filter="A11_") -> pd.DataFrame:
        """Read AE33 data file into a pd.DataFrame

        Args:
            file (str): full path to file
            sep (str, optional): field separator used in file. Defaults to "|".
            type (str, optional): Archive type. Defaults to ".tar.gz"
            filter (str, optional): Filter for file inside archive. Defaults to "A11_"
        Returns:
            pd.DataFrame: DataFrame with dtm and source columns added to data
            
        """
        try: 
            cols = ["A11a", "STN", "EPOCH", "dtm", "Q_A11", "PCT_A11", 
                        "X1c_A11", "X2c_A11", "X3c_A11", "X4c_A11", "X5c_A11", "X6c_A11", "X7c_A11", 
                        "ZIr1_A11", "ZIr2_A11", "ZIr3_A11", "ZIr4_A11", "ZIr5_A11", "ZIr6_A11", "ZIr7_A11", 
                        "Ipz1_A11", "Ipz2_A11", "Ipz3_A11", "Ipz4_A11", "Ipz5_A11", "Ipz6_A11", "Ipz7_A11", 
                        "Ip1_A11", "Ip2_A11", "Ip3_A11", "Ip4_A11", "Ip5_A11", "Ip6_A11", "Ip7_A11", "Ifz1_A11", "Ifz2_A11", "Ifz3_A11", "Ifz4_A11", "Ifz5_A11", "Ifz6_A11", "Ifz7_A11", 
                        "If1_A11", "If2_A11", "If3_A11", "If4_A11", "If5_A11", "If6_A11", "If7_A11"]
            na_values = ["A11a", "ZZZ", "0", "9999-99-99T99:99:99Z", "0.00", "0.00", "000000", "000000", "000000", "000000", "000000", "000000", "000000", "000.000", "000.000", "000.000", "000.000", "000.000", "000.000", "000.000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000", "00.0000"]
            varfmt = ["A11a", "%s", "%u", "%04d-%02d-%02dT%02d:%02d:%02dZ", "*@01.2f", "*@01.2f", "*@06.0f", "*@06.0f", "*@06.0f", "*@06.0f", "*@06.0f", "*@06.0f", "*@06.0f", "*@03.3f", "*@03.3f", "*@03.3f", "*@03.3f", "*@03.3f", "*@03.3f", "*@03.3f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f", "*@02.4f"]

            with tarfile.open(file, "r:*") as tar:
                files = tar.getnames()
                print(files)
                
                fh = [s for s in files if "A11_" in s][0]
                print(fh)
                
                df = pd.read_csv(tar.extractfile(fh), header=None,skiprows=100, names=cols, sep=sep)
                df["dtm"] = pd.to_datetime(df["dtm"], utc=True)

                df['source'] = fh
                

            return df
        
        except Exception as err:
            print(err)

In [ ]:
# read data of every year and save as .txt files

years = [2016,2017,2020,2021,2022,2023]

for year in years:

    path = "/product_data/data/pay/Kenya/MKN/incoming/aerosol/"+str(year)+"/"

    files = []

    # get all paths of files
    for file in os.walk(path):
        if "mkn_" in str(file):
            for f in file[2]:
                files.append(str(file[0])+"/"+str(f))
    
    # extract all files and create data frame
    df = extract_file(file=os.path.join(path, files[0]))
    for file in files[1:]:
        df = pd.concat([df, extract_file(file=file)])

    # remove extremes
    df.set_index("dtm", inplace=True)
    N = [1,2,3,4,5,6,7]
    for n in N:
        lower = df[f"X{n}c_A11"].quantile(q=0.01)
        upper = df[f"X{n}c_A11"].quantile(q=1-0.01)
        df.loc[(df[f"X{n}c_A11"] < lower) | (df[f"X{n}c_A11"] > upper), f"X{n}c_A11"] = None
    df.reset_index(inplace=True)

    # save data frame
    df.to_csv(os.getcwd()+"/data/aerosol_MCH/ae31/"+"ae31_A11_"+str(year)+".txt", sep=",")

In [ ]:
# save data frame

df.to_csv(os.getcwd()+"/data/aerosol_MCH/ae31/"+"ae31_A11.txt", sep=",")

In [ ]:
#### AE33 Files ####

def extract_file(file: str, sep="|") -> pd.DataFrame:
        """Read AE33 data file into a pd.DataFrame

        Args:
            file (str): full path to file
            sep (str, optional): field separator used in file. Defaults to "|".

        Returns:
            pd.DataFrame: DataFrame with dtm and source columns added to data

        """
        try:
            cols = ["Inst_SN", "row_id", "dtm_1", 
                    "dtm", "unclear", "dtm_2", 
                    "RefCh1", "Sen1Ch1", "Sen2Ch1", 
                    "RefCh2", "Sen1Ch2", "Sen2Ch2", 
                    "RefCh3", "Sen1Ch3", "Sen2Ch3", 
                    "RefCh4", "Sen1Ch4", "Sen2Ch4", 
                    "RefCh5", "Sen1Ch5", "Sen2Ch5", 
                    "RefCh6", "Sen1Ch6", "Sen2Ch6", 
                    "RefCh7", "Sen1Ch7", "Sen2Ch7", 
                    "BC11", "BC12", "BC1", 
                    "BC21", "BC22", "BC2", 
                    "BC31", "BC32", "BC3", 
                    "BC41", "BC42", "BC4", 
                    "BC51", "BC52", "BC5", 
                    "BC61", "BC62", "BC6", 
                    "BC71", "BC72", "BC7", 
                    "K1", "K2", "K3", "K4", "K5", "K6", "K7", 
                    "unclear_2", # "BB"
                    "Pres", "Temp", 
                    "Flow1", "Flow2", "FlowC", 
                    "Temp_1", "Temp_2","Temp_3",
                    # "ContTemp", "SupplyTemp", "LedTemp",
                    "Stat_1", "Stat_2", "Stat_3", "Stat_4", "Stat_5", 
                    # "Status", "ContStatus", "DetectStatus", "LedStatus", "ValveStatus", 
                    "TapeAdvCount", "unknown_2", "unknown_3", "unknown_4", "unknown_5"
                    # "ID_com1", "ID_com2", "ID_com3", "fields_i"
                    ]

            df = pd.read_csv(file, sep=sep, header=None, names=cols)
            df["dtm"] = pd.to_datetime(df["dtm"], utc=True)
            # remove all data prior to the deployment date of instrument
            df.drop(df[df["dtm"] < pd.to_datetime("2022-12-09", utc=True)].index, inplace=True)

            df["dtm_1"] = pd.to_datetime(df["dtm_1"])
            df["dtm_2"] = pd.to_datetime(df["dtm_2"])
            
            return df
        
        except Exception as err:
            print(err)

In [ ]:
def remove_extremes(df: pd.DataFrame, q=0.01, index="dtm") -> pd.DataFrame:
    try:
        df.set_index(index, inplace=True)
        N = [1,2,3,4,5,6,7]
        for n in N:
            lower = df.quantile(q, numeric_only=True)
            upper = df.quantile(1-q, numeric_only=True)
            df.loc[(df[f"BC{n}"] < lower[f"BC{n}"]) | (df[f"BC{n}"] > upper[f"BC{n}"]), f"BC{n}"] = None
            df.reset_index(inplace=True)        
        return df    

    except Exception as err:
        print(err)

In [ ]:
# read data of every year and save as .txt files

years = [2022,2023]

for year in years: 
    path = "/product_data/data/pay/Kenya/MKN/incoming/ae33/data/"+str(year)+"/"

    files = []

    #find all files
    for file in os.walk(path):
        if "ae33-" in str(file):
            for f in file[2]:
                files.append(str(file[0])+"/"+str(f))
    
    # extract files and save to data frame
    df = extract_file(file= files[0])
    for file in files[1:]:
        df = pd.concat([df, extract_file(file=file)])

    remove_extremes(df)

    df.to_csv(os.getcwd()+"/data/aerosol_MCH/ae33/"+"ae33_"+str(year)+".txt", sep=",")

In [ ]:
##### AE31 S11 Files #####

def extract_file(file: str, sep=",") -> pd.DataFrame:
        """Read AE33 data file into a pd.DataFrame

        Args:
            file (str): full path to file
            sep (str, optional): field separator used in file. Defaults to "|".
        Returns:
            pd.DataFrame: DataFrame with dtm and source columns added to data
        """
        try:
        
            cols = ["S11a","STN","EPOCH","dtm","F1_S11","F2_S11","BsB_S11","BsG_S11","BsR_S11","BbsB_S11","BbsG_S11","BbsR_S11","T1_S11","T2_S11","U_S11","P_S11"]
            na_values = ["S11a","ZZZ","0","9999-99-99T99:99:99Z","FFFF","FFFF","9999.99","9999.99","9999.99","9999.99","9999.99","9999.99","999.9","999.9","999.9","9999.9"]
            varfmt = ["S11a","%s","%u","%04d-%02d-%02dT%02d:%02d:%02dZ","%04X","%04X","*@04.2f","*@04.2f","*@04.2f","*@04.2f","*@04.2f","*@04.2f","*@03.1f","*@03.1f","*@03.1f","*@04.1f"]

            with tarfile.open(file, "r:*") as tar:
                files = tar.getnames()
                fh = [s for s in files if "S11_" in s]
                print(fh)
        
                df = pd.DataFrame(columns=cols, dtype=object)
                for f in fh:
                    df_new = pd.read_csv(tar.extractfile(str(f)), skiprows=354, sep=sep, header=None, names=cols, error_bad_lines=False)
                    df_new['source'] = f
                    df_new["dtm"] = pd.to_datetime(df_new["dtm"], utc=True)
                    df = pd.concat([df, df_new])
    
            return df
        
        except Exception as err:
            print(err)

In [ ]:
# read data of every year and save as .txt files

years = [2016,2017,2020,2021,2022,2023]

for year in years:
    path = "/product_data/data/pay/Kenya/MKN/incoming/aerosol/"+str(year)+"/"

    files = []

    # get path to all files
    for file in os.walk(path):
        if "mkn_" in str(file):
            for f in file[2]:
                files.append(str(file[0])+"/"+str(f))

    # read files, create data frame
    df = extract_file(file=os.path.join(path, files[0]))
    for file in files[1:]:
        df = pd.concat([df, extract_file(file=file)])
    
    # save data frame
    df.to_csv(os.getcwd()+"/data/aerosol_MCH/ae31/"+"ae31_S11a_" +str(year)+".txt", sep=",")

In [ ]:
#### MCH ozone  #####

def extract_file(file: str) -> pd.DataFrame:
        """Read AE33 data file into a pd.DataFrame

        Returns:
            pd.DataFrame: DataFrame with dtm and source columns added to data

        """
        try:
            cols = ["pcdate", "pctime", "time", 
                    "date", "o3", "flags", 
                    "cellai", "cellbi", "bncht", 
                    "lmpt", "o3lt", "flowa", 
                    "flowb", "pres"
                    ]
            
            df = pd.read_csv(file, names=cols, skiprows=1, sep=" ")
      
            return df
        
        except Exception as err:
            print(err)

In [6]:
# read data of every year and save as .txt files

years = [2022,2023]

for year in years: 
    path = "/product_data/data/pay/Kenya/MKN/incoming/tei49c/"+str(year)+"/"
    print(path)

    files = []

    # get path of all files
    for file in os.walk(path):
        if ".zip" in str(file):
            for f in file[2]:
                files.append(str(file[0])+"/"+str(f))

    # extract files and save to data frame
    df = extract_file(file=os.path.join(path, files[0]))
    for file in files[1:]:
        df = pd.concat([df, extract_file(file=file)])

    # save data frame
    df.to_csv(os.getcwd()+"/data/ozone_MCH/ozone_"+str(year)+".txt")

/product_data/data/pay/Kenya/MKN/incoming/tei49c/2022
/product_data/data/pay/Kenya/MKN/incoming/tei49c/2023
